# PostgreSQL · 50 задач перед контрольной

**Пиши → запускай Shift+Enter → исправляй → проверяй снова.**

Задачи **1–6** повторяют запросы **3–8** из твоего примера. Затем сложность растет до окон, множеств и серий дат.

1. Запусти ячейку подключения ниже. В Docker база запускается автоматически вместе с контейнером.
2. В ячейке `%%check N` замени комментарий своим SQL. Первая строка должна остаться.
3. Для исследования данных используй `%%sql`. Подсказка: `%hint N`; эталон: `%solution N`; прогресс: `progress()`.

Ответы сохраняются в `answers/NN.sql`. Проверка сравнивает результаты на 5 наборах, включая пустой. **Таблицы пиши без `public.`**. Колонки — точно как в условии. Если сортировка не указана, порядок строк не важен, но дубликаты важны. Числа сравниваются после округления до 6 знаков.

**Моделирование М1/М2:** отдельные задания и чек-лист в `models/README.md`. **Пробная контрольная:** `exam.ipynb`.

In [ ]:
from trainer import *
install()

## Осмотр данных

Входная роль — только чтение; запрос не испортит учебные таблицы. Предел одного запроса — 4 секунды. `%%sql check_1` переключает предварительный просмотр на альтернативный набор.

In [ ]:
%%sql
SELECT id, title, budget, status, completed_at
FROM projects
ORDER BY id
LIMIT 10;

## Карта базы



### Словарь колонок

| Таблица | Колонки |
|---|---|
| `users` | `id`, `name`, `registered_at` |
| `customers` | `user_id` (PK и FK → users.id) |
| `freelancers` | `user_id` (PK и FK), `specialization`, `hourly_rate` |
| `projects` | `id`, `customer_id`, `title`, `description`, `budget`, `status`, `created_at`, `completed_at` |
| `bids` | `id`, `project_id`, `freelancer_id`, `amount`, `proposed_completion_date`, `status`, `submitted_at`, `accepted_at`, `rejected_at` |
| `reviews` | `project_id` (PK и FK), `rating`, `comment`, `created_at` |

Проекты: **открыт / в работе / завершен**. Заявки: **подана / принята / отклонена**. Исполнитель — фрилансер принятой заявки. Одна заявка человека на проект; максимум один победитель; один отзыв на завершенный проект.

Даты событий — timestamp, обещанный день — date, деньги — numeric. Часовая ставка (`hourly_rate`), бюджет проекта (`budget`) и предложенная цена (`amount`) — разные величины. Имена пользователей могут повторяться.

В спорных формулировках приняты явные правила: непустые проекты в №2; все проекты в знаменателе №3; 10% от меньшей суммы в №6. На контрольной эти трактовки лучше обозначить в ответе.

## I · Запросы из примера контрольной

### 01. Python и заявки за 2023 год

Сложность **2/5**.

Аналог задания 3. Найдите Python-фрилансеров, подавших строго больше 3 заявок в 2023 году. Одна строка на человека, даже если имена совпадают. Имя и точное количество заявок.

**Колонки в указанном порядке:** `name`, `bid_count`.

Порядок строк не важен. Подсказка по запросу: `%hint 1`.

In [ ]:
%%check 1
-- Напиши один SELECT или WITH ... SELECT.


### 02. Только дорогие специалисты

Сложность **3/5**.

Аналог задания 4. Проекты с бюджетом > 50000, у которых есть хотя бы одна заявка и нет заявок от фрилансеров с hourly_rate < 1500. Ровно 1500 допустимо.

**Колонки в указанном порядке:** `project_id`, `title`.

Порядок строк не важен. Подсказка по запросу: `%hint 2`.

In [ ]:
%%check 2
-- Напиши один SELECT или WITH ... SELECT.


### 03. Среднее по всем проектам

Сложность **3/5**.

Аналог задания 5. Для каждой имеющейся специализации посчитайте среднее число ее заявок на проект. Знаменатель — ВСЕ проекты базы, включая проекты без заявок этой специализации. Специализации без заявок тоже выводите с 0. Если проектов вообще нет, среднее NULL. Не округляйте.

**Колонки в указанном порядке:** `specialization`, `avg_bids`.

Порядок строк не важен. Подсказка по запросу: `%hint 3`.

In [ ]:
%%check 3
-- Напиши один SELECT или WITH ... SELECT.


### 04. Завершенные проекты прошлого месяца

Сложность **3/5**.

Аналог задания 6. Для специализаций исполнителей проектов, завершенных в прошлом календарном месяце относительно CURRENT_DATE, выведите число проектов и сумму их БЮДЖЕТОВ. Исполнитель определяется принятой заявкой. Специализации без таких проектов не выводите.

**Колонки в указанном порядке:** `specialization`, `project_count`, `total_budget`.

Порядок строк не важен. Подсказка по запросу: `%hint 4`.

In [ ]:
%%check 4
-- Напиши один SELECT или WITH ... SELECT.


### 05. Три прибыльные специализации

Сложность **3/5**.

Аналог задания 7. Верните ровно три специализации с наибольшей суммой сумм принятых заявок за все время (или меньше, если их меньше трех). Только специализации с принятыми заявками. Сортировка total_amount DESC, при равенстве specialization ASC.

**Колонки в указанном порядке:** `specialization`, `total_amount`.

**Порядок строк проверяется.** Подсказка по запросу: `%hint 5`.

In [ ]:
%%check 5
-- Напиши один SELECT или WITH ... SELECT.


### 06. Пары с близкими ценами

Сложность **3/5**.

Аналог задания 8. Найдите пары фрилансеров с заявками на общий проект, где max(amount) <= min(amount) * 1.10. То есть разница не более 10% МЕНЬШЕЙ цены. Выводите пару ID по возрастанию, один раз за все проекты; проект в результат не включайте.

**Колонки в указанном порядке:** `freelancer_id_1`, `freelancer_id_2`.

Порядок строк не важен. Подсказка по запросу: `%hint 6`.

In [ ]:
%%check 6
-- Напиши один SELECT или WITH ... SELECT.


## II · JOIN, группировки и исключения

### 07. Проекты без заявок

Сложность **2/5**.

Выведите ID и название всех проектов, на которые не поступило ни одной заявки.

**Колонки в указанном порядке:** `project_id`, `title`.

Порядок строк не важен. Подсказка по запросу: `%hint 7`.

In [ ]:
%%check 7
-- Напиши один SELECT или WITH ... SELECT.


### 08. Количество заявок у каждого фрилансера

Сложность **2/5**.

Все фрилансеры, включая тех, у кого 0 заявок. Выведите ID, имя и количество заявок.

**Колонки в указанном порядке:** `freelancer_id`, `name`, `bid_count`.

Порядок строк не важен. Подсказка по запросу: `%hint 8`.

In [ ]:
%%check 8
-- Напиши один SELECT или WITH ... SELECT.


### 09. Дешевле собственного среднего

Сложность **2/5**.

Найдите заявки со стоимостью строго ниже среднего всех заявок того же фрилансера. Выведите ID заявки и ее сумму.

**Колонки в указанном порядке:** `bid_id`, `amount`.

Порядок строк не важен. Подсказка по запросу: `%hint 9`.

In [ ]:
%%check 9
-- Напиши один SELECT или WITH ... SELECT.


### 10. Экономия на выполненных проектах

Сложность **2/5**.

Для каждого завершенного проекта выведите бюджет минус сумму принятой заявки. Отрицательная экономия допустима.

**Колонки в указанном порядке:** `project_id`, `savings`.

Порядок строк не важен. Подсказка по запросу: `%hint 10`.

In [ ]:
%%check 10
-- Напиши один SELECT или WITH ... SELECT.


### 11. Воронка заказчиков

Сложность **3/5**.

Для каждого заказчика, включая заказчиков без проектов, выведите число всех, открытых, выполняемых и завершенных проектов.

**Колонки в указанном порядке:** `customer_id`, `total_count`, `open_count`, `active_count`, `completed_count`.

Порядок строк не важен. Подсказка по запросу: `%hint 11`.

In [ ]:
%%check 11
-- Напиши один SELECT или WITH ... SELECT.


### 12. Доля принятых заявок

Сложность **3/5**.

Для каждого фрилансера: количество всех заявок, принятых заявок, процент принятых от всех (0..100), округленный до 2 знаков. При отсутствии заявок процент 0.

**Колонки в указанном порядке:** `freelancer_id`, `bid_count`, `accepted_count`, `acceptance_pct`.

Порядок строк не важен. Подсказка по запросу: `%hint 12`.

In [ ]:
%%check 12
-- Напиши один SELECT или WITH ... SELECT.


### 13. Среднее только по участвовавшим проектам

Сложность **3/5**.

Для каждой специализации с заявками найдите среднее число ее заявок по тем проектам, где есть хотя бы одна заявка этой специализации. Не округляйте. Сравните смысл с задачей 3.

**Колонки в указанном порядке:** `specialization`, `avg_bids`.

Порядок строк не важен. Подсказка по запросу: `%hint 13`.

In [ ]:
%%check 13
-- Напиши один SELECT или WITH ... SELECT.


### 14. Разнообразие участников

Сложность **2/5**.

Найдите проекты с заявками минимум от трех разных специализаций. Выведите ID и число специализаций.

**Колонки в указанном порядке:** `project_id`, `specialization_count`.

Порядок строк не важен. Подсказка по запросу: `%hint 14`.

In [ ]:
%%check 14
-- Напиши один SELECT или WITH ... SELECT.


### 15. Работали с Python и дизайнерами

Сложность **3/5**.

Заказчики, у которых есть завершенный проект с исполнителем Python и есть завершенный проект с исполнителем Дизайн. Выведите ID заказчика.

**Колонки в указанном порядке:** `customer_id`.

Порядок строк не важен. Подсказка по запросу: `%hint 15`.

In [ ]:
%%check 15
-- Напиши один SELECT или WITH ... SELECT.


### 16. Только Python

Сложность **3/5**.

Найдите проекты хотя бы с одной заявкой, у которых все участники специализируются на Python.

**Колонки в указанном порядке:** `project_id`.

Порядок строк не важен. Подсказка по запросу: `%hint 16`.

In [ ]:
%%check 16
-- Напиши один SELECT или WITH ... SELECT.


### 17. Все заявки дороже бюджета

Сложность **3/5**.

Проекты с заявками, где каждая предложенная сумма строго больше бюджета проекта.

**Колонки в указанном порядке:** `project_id`.

Порядок строк не важен. Подсказка по запросу: `%hint 17`.

In [ ]:
%%check 17
-- Напиши один SELECT или WITH ... SELECT.


### 18. Конкурент дешевле победителя

Сложность **3/5**.

Выведите ID проектов, где существует отклоненная заявка дешевле принятой. Каждый проект один раз.

**Колонки в указанном порядке:** `project_id`.

Порядок строк не важен. Подсказка по запросу: `%hint 18`.

In [ ]:
%%check 18
-- Напиши один SELECT или WITH ... SELECT.


## III · Окна, ранги и даты

### 19. Две самые дешевые заявки

Сложность **3/5**.

В каждом проекте выберите не более двух заявок: amount ASC, при равенстве id ASC. Выведите ID проекта, ID заявки, сумму и номер позиции (1 или 2). Общий порядок строк не важен.

**Колонки в указанном порядке:** `project_id`, `bid_id`, `amount`, `position`.

Порядок строк не важен. Подсказка по запросу: `%hint 19`.

In [ ]:
%%check 19
-- Напиши один SELECT или WITH ... SELECT.


### 20. Два минимальных уровня цен

Сложность **4/5**.

В каждом проекте найдите заявки, относящиеся к двум наименьшим различным ценам. При равенстве включайте всех. Выведите уровень 1 или 2.

**Колонки в указанном порядке:** `project_id`, `bid_id`, `amount`, `price_rank`.

Порядок строк не важен. Подсказка по запросу: `%hint 20`.

In [ ]:
%%check 20
-- Напиши один SELECT или WITH ... SELECT.


### 21. Рейтинг доходов внутри специализации

Сложность **4/5**.

Все фрилансеры, включая без побед. Доход — сумма принятых заявок, иначе 0. Присвойте DENSE_RANK по доходу DESC внутри специализации.

**Колонки в указанном порядке:** `freelancer_id`, `specialization`, `revenue`, `revenue_rank`.

Порядок строк не важен. Подсказка по запросу: `%hint 21`.

In [ ]:
%%check 21
-- Напиши один SELECT или WITH ... SELECT.


### 22. Накопленный доход

Сложность **4/5**.

Для каждой принятой заявки выведите ее ID, ID фрилансера, accepted_at и накопленную сумму его принятых заявок. Порядок внутри истории: accepted_at ASC, id ASC.

**Колонки в указанном порядке:** `bid_id`, `freelancer_id`, `accepted_at`, `running_revenue`.

Порядок строк не важен. Подсказка по запросу: `%hint 22`.

In [ ]:
%%check 22
-- Напиши один SELECT или WITH ... SELECT.


### 23. Перерыв между заявками

Сложность **4/5**.

Для каждой заявки выведите ID, ID фрилансера и разницу в целых календарных днях между submitted_at::date текущей и предыдущей заявки этого фрилансера. Порядок submitted_at, id. Для первой заявки gap_days = NULL.

**Колонки в указанном порядке:** `bid_id`, `freelancer_id`, `gap_days`.

Порядок строк не важен. Подсказка по запросу: `%hint 23`.

In [ ]:
%%check 23
-- Напиши один SELECT или WITH ... SELECT.


### 24. Последняя заявка каждого

Сложность **3/5**.

Для фрилансеров с заявками найдите последнюю заявку по submitted_at DESC, id DESC. Выведите freelancer_id, bid_id, submitted_at, amount.

**Колонки в указанном порядке:** `freelancer_id`, `bid_id`, `submitted_at`, `amount`.

Порядок строк не важен. Подсказка по запросу: `%hint 24`.

In [ ]:
%%check 24
-- Напиши один SELECT или WITH ... SELECT.


### 25. Отклонение цены от среднего проекта

Сложность **3/5**.

Для каждой заявки выведите ее ID, среднюю цену заявок ее проекта и отклонение своей цены от этого среднего. Оба результата округлите до 2 знаков; отклонение считайте от неокругленного среднего.

**Колонки в указанном порядке:** `bid_id`, `project_avg`, `difference`.

Порядок строк не важен. Подсказка по запросу: `%hint 25`.

In [ ]:
%%check 25
-- Напиши один SELECT или WITH ... SELECT.


### 26. Медианная цена

Сложность **4/5**.

Для каждого проекта с заявками посчитайте непрерывную медиану сумм (для четного количества — среднее двух центральных). Округлите до 2 знаков.

**Колонки в указанном порядке:** `project_id`, `median_amount`.

Порядок строк не важен. Подсказка по запросу: `%hint 26`.

In [ ]:
%%check 26
-- Напиши один SELECT или WITH ... SELECT.


### 27. Календарь заявок за 2023 год

Сложность **4/5**.

Выведите все 12 месяцев 2023 года, даже пустые: month — дата первого дня месяца, bid_count — число поданных заявок. Сортировка month ASC.

**Колонки в указанном порядке:** `month`, `bid_count`.

**Порядок строк проверяется.** Подсказка по запросу: `%hint 27`.

In [ ]:
%%check 27
-- Напиши один SELECT или WITH ... SELECT.


### 28. Месячная выручка и изменение

Сложность **4/5**.

За последние 6 ПОЛНЫХ месяцев (текущий исключается) выведите month (date), сумму принятых заявок по accepted_at и изменение относительно предыдущего месяца в этом шестимесячном окне. Пустой месяц = 0; первое изменение = NULL. Сортировка month ASC.

**Колонки в указанном порядке:** `month`, `revenue`, `delta`.

**Порядок строк проверяется.** Подсказка по запросу: `%hint 28`.

In [ ]:
%%check 28
-- Напиши один SELECT или WITH ... SELECT.


### 29. Скользящее среднее за три месяца

Сложность **4/5**.

Для 12 месяцев 2023 года выведите month (date), bid_count и среднее количества заявок за текущий и два предыдущих месяца в пределах 2023. Для января окно из 1 месяца, февраля из 2. Округлите среднее до 2 знаков. Сортировка month ASC.

**Колонки в указанном порядке:** `month`, `bid_count`, `moving_avg`.

**Порядок строк проверяется.** Подсказка по запросу: `%hint 29`.

In [ ]:
%%check 29
-- Напиши один SELECT или WITH ... SELECT.


### 30. Победа с первой попытки

Сложность **4/5**.

Фрилансеры, у которых самая ранняя заявка по (submitted_at, id) оказалась принятой. Выведите freelancer_id и bid_id этой заявки.

**Колонки в указанном порядке:** `freelancer_id`, `bid_id`.

Порядок строк не важен. Подсказка по запросу: `%hint 30`.

In [ ]:
%%check 30
-- Напиши один SELECT или WITH ... SELECT.


## IV · Множества и сложные условия

### 31. Все открытые проекты заказчика

Сложность **5/5**.

Найдите пары (customer_id, freelancer_id), где у заказчика есть хотя бы один открытый проект, а фрилансер подал заявку на КАЖДЫЙ открытый проект этого заказчика. Статус заявки любой.

**Колонки в указанном порядке:** `customer_id`, `freelancer_id`.

Порядок строк не важен. Подсказка по запросу: `%hint 31`.

In [ ]:
%%check 31
-- Напиши один SELECT или WITH ... SELECT.


### 32. Одинаковые наборы проектов

Сложность **5/5**.

Найдите пары разных фрилансеров с одинаковым НЕПУСТЫМ множеством проектов, куда они подавали заявки. ID внутри пары по возрастанию; пара один раз.

**Колонки в указанном порядке:** `freelancer_id_1`, `freelancer_id_2`.

Порядок строк не важен. Подсказка по запросу: `%hint 32`.

In [ ]:
%%check 32
-- Напиши один SELECT или WITH ... SELECT.


### 33. Не менее трех общих проектов

Сложность **4/5**.

Для каждой пары фрилансеров с минимум тремя общими проектами выведите ID по возрастанию и число общих проектов.

**Колонки в указанном порядке:** `freelancer_id_1`, `freelancer_id_2`, `common_projects`.

Порядок строк не важен. Подсказка по запросу: `%hint 33`.

In [ ]:
%%check 33
-- Напиши один SELECT или WITH ... SELECT.


### 34. Единственная самая дешевая заявка

Сложность **4/5**.

Для каждого проекта найдите заявку, которая строго дешевле ВСЕХ остальных. При равном минимуме проект исключите. Единственная заявка в проекте подходит.

**Колонки в указанном порядке:** `project_id`, `bid_id`, `amount`.

Порядок строк не важен. Подсказка по запросу: `%hint 34`.

In [ ]:
%%check 34
-- Напиши один SELECT или WITH ... SELECT.


### 35. Победитель по средней оценке

Сложность **4/5**.

Для каждой специализации найдите фрилансеров с максимальной средней оценкой при наличии минимум двух отзывов. Включайте всех при равенстве неокругленных средних. Выведите среднюю оценку, округленную до 2 знаков.

**Колонки в указанном порядке:** `specialization`, `freelancer_id`, `avg_rating`.

Порядок строк не важен. Подсказка по запросу: `%hint 35`.

In [ ]:
%%check 35
-- Напиши один SELECT или WITH ... SELECT.


### 36. Повторные заказчики

Сложность **3/5**.

Для пар заказчик–фрилансер с минимум двумя завершенными совместными проектами выведите ID обоих, количество проектов и сумму принятых заявок.

**Колонки в указанном порядке:** `customer_id`, `freelancer_id`, `project_count`, `total_amount`.

Порядок строк не важен. Подсказка по запросу: `%hint 36`.

In [ ]:
%%check 36
-- Напиши один SELECT или WITH ... SELECT.


### 37. Второй различный бюджет

Сложность **4/5**.

Для каждого заказчика со минимум двумя различными бюджетами проектов выведите второй по величине различный бюджет.

**Колонки в указанном порядке:** `customer_id`, `second_budget`.

Порядок строк не важен. Подсказка по запросу: `%hint 37`.

In [ ]:
%%check 37
-- Напиши один SELECT или WITH ... SELECT.


### 38. Завершены, но без отзывов

Сложность **3/5**.

Для каждого заказчика с завершенными проектами выведите количество завершенных проектов, число из них без отзыва и процент без отзыва, округленный до 2 знаков.

**Колонки в указанном порядке:** `customer_id`, `completed_count`, `missing_reviews`, `missing_pct`.

Порядок строк не важен. Подсказка по запросу: `%hint 38`.

In [ ]:
%%check 38
-- Напиши один SELECT или WITH ... SELECT.


### 39. Просрочки исполнителей

Сложность **4/5**.

Для каждого фрилансера с завершенными проектами: число завершенных проектов, число просроченных и среднее число дней просрочки ТОЛЬКО просроченных. Просрочка = completed_at::date > proposed_completion_date. Среднее округлите до 2 знаков; если просрочек нет, NULL.

**Колонки в указанном порядке:** `freelancer_id`, `completed_count`, `late_count`, `avg_late_days`.

Порядок строк не важен. Подсказка по запросу: `%hint 39`.

In [ ]:
%%check 39
-- Напиши один SELECT или WITH ... SELECT.


### 40. Выигрыши за месяц после регистрации

Сложность **4/5**.

Для каждой специализации выведите общее количество фрилансеров и сколько из них имеют хотя бы одну принятую заявку с accepted_at в [registered_at, registered_at + 1 month). Месяц календарный, не 30 суток.

**Колонки в указанном порядке:** `specialization`, `freelancer_count`, `early_winners`.

Порядок строк не важен. Подсказка по запросу: `%hint 40`.

In [ ]:
%%check 40
-- Напиши один SELECT или WITH ... SELECT.


## V · Продвинутые отчеты

### 41. Время ожидания решения

Сложность **3/5**.

Для каждой специализации с рассмотренными заявками посчитайте среднее время в часах от подачи до принятия/отклонения; поданные без решения исключите. Округлите до 2 знаков.

**Колонки в указанном порядке:** `specialization`, `avg_hours`.

Порядок строк не важен. Подсказка по запросу: `%hint 41`.

In [ ]:
%%check 41
-- Напиши один SELECT или WITH ... SELECT.


### 42. Серии активных месяцев

Сложность **5/5**.

По календарю 2023 года для каждого фрилансера найдите максимальную длину серии последовательных месяцев, в каждом из которых была хотя бы одна подача заявки. Фрилансеров без заявок в 2023 выводите с 0.

**Колонки в указанном порядке:** `freelancer_id`, `longest_streak`.

Порядок строк не важен. Подсказка по запросу: `%hint 42`.

In [ ]:
%%check 42
-- Напиши один SELECT или WITH ... SELECT.


### 43. Пропущенные месяцы между первым и последним

Сложность **5/5**.

Для каждого фрилансера выведите месяцы 2023 года БЕЗ заявок между его первым и последним активным месяцем в 2023 включительно. month — date первого числа. Без активности в 2023 строк нет.

**Колонки в указанном порядке:** `freelancer_id`, `month`.

Порядок строк не важен. Подсказка по запросу: `%hint 43`.

In [ ]:
%%check 43
-- Напиши один SELECT или WITH ... SELECT.


### 44. Доля в доходе специализации

Сложность **4/5**.

Все фрилансеры: собственный доход от принятых заявок и доля в сумме доходов специализации в процентах (2 знака). При нулевом доходе всей специализации share_pct = 0.

**Колонки в указанном порядке:** `freelancer_id`, `specialization`, `revenue`, `share_pct`.

Порядок строк не важен. Подсказка по запросу: `%hint 44`.

In [ ]:
%%check 44
-- Напиши один SELECT или WITH ... SELECT.


### 45. Минимальный префикс на 80% выручки

Сложность **5/5**.

Среди фрилансеров с положительным доходом от принятых заявок выберите минимальный префикс списка revenue DESC, freelancer_id ASC, который покрывает не менее 80% общей выручки. Включайте строку, пересекающую порог. Выведите ID, revenue и cumulative_pct (2 знака). Сортировка как в списке.

**Колонки в указанном порядке:** `freelancer_id`, `revenue`, `cumulative_pct`.

**Порядок строк проверяется.** Подсказка по запросу: `%hint 45`.

In [ ]:
%%check 45
-- Напиши один SELECT или WITH ... SELECT.


### 46. Широкая таблица заявок по статусам

Сложность **3/5**.

Для каждой специализации, даже без заявок, выведите количество поданных, принятых и отклоненных заявок и сумму принятых (0 при отсутствии).

**Колонки в указанном порядке:** `specialization`, `pending_count`, `accepted_count`, `rejected_count`, `accepted_amount`.

Порядок строк не важен. Подсказка по запросу: `%hint 46`.

In [ ]:
%%check 46
-- Напиши один SELECT или WITH ... SELECT.


### 47. Состав специализаций проекта

Сложность **3/5**.

Для каждого проекта, включая без заявок, выведите отсортированные по алфавиту уникальные специализации участников, соединенные через запятую и пробел. Для пустого проекта — пустая строка.

**Колонки в указанном порядке:** `project_id`, `specializations`.

Порядок строк не важен. Подсказка по запросу: `%hint 47`.

In [ ]:
%%check 47
-- Напиши один SELECT или WITH ... SELECT.


### 48. Фрилансер выше среднего коллег

Сложность **4/5**.

Найдите фрилансеров, чей hourly_rate строго больше среднего hourly_rate ДРУГИХ фрилансеров той же специализации. Если коллег нет, исключите. Выведите ID, ставку и среднее коллег, округленное до 2 знаков; сравнивайте до округления.

**Колонки в указанном порядке:** `freelancer_id`, `hourly_rate`, `peers_avg`.

Порядок строк не важен. Подсказка по запросу: `%hint 48`.

In [ ]:
%%check 48
-- Напиши один SELECT или WITH ... SELECT.


### 49. Минимальная цена и бюджет по заказчикам

Сложность **4/5**.

Для каждого заказчика выведите число проектов, сумму бюджетов и сумму минимальных предложенных цен по его проектам. Проект без заявок дает 0 в последнюю сумму. Заказчик без проектов: все три показателя 0. Не допускайте умножения бюджетов из-за JOIN.

**Колонки в указанном порядке:** `customer_id`, `project_count`, `total_budget`, `total_min_bid`.

Порядок строк не важен. Подсказка по запросу: `%hint 49`.

In [ ]:
%%check 49
-- Напиши один SELECT или WITH ... SELECT.


### 50. Сводный отчет без двойного счета

Сложность **5/5**.

Для каждой специализации выведите число фрилансеров, число всех заявок, число уникальных проектов с заявками, число завершенных проектов ее исполнителей, доход по всем принятым заявкам и средний рейтинг по отзывам выполненных проектов. Специализации без активности включите, числовые счетчики/доход = 0, рейтинг = NULL; рейтинг округлите до 2 знаков.

**Колонки в указанном порядке:** `specialization`, `freelancer_count`, `bid_count`, `project_count`, `completed_count`, `revenue`, `avg_rating`.

Порядок строк не важен. Подсказка по запросу: `%hint 50`.

In [ ]:
%%check 50
-- Напиши один SELECT или WITH ... SELECT.


## Прогресс

Сохраняется результат последней проверки каждой задачи. Эталоны можно открыть через `%solution N` после самостоятельной попытки.

In [ ]:
progress()